In [ ]:
!pip install "z3-solver"
from z3 import *


## Задача 1. Яблоки

У Маши было некоторое количество яблок. Петя дал ей ещё 2 яблока, а затем мама дала ещё столько яблок, сколько было у Маши сначала. Всего у Маши стало 10 яблок. Сколько яблок было у неё сначала?

In [ ]:
x = Int("apples")

goal = Goal()
goal.add(x + 2 + x == 10)

print("Упрощённая цель:", Tactic("simplify")(goal))

solver = Then("simplify", "solve-eqs").solver()
solver.add(x + 2 + x == 10)
print("Статус:", solver.check())
print("Яблок сначала:", solver.model()[x])


## Задача 2. Три выключателя

У Маши есть три выключателя. Каждый выключатель может быть выключен или включён. Известно, что включены ровно два выключателя. Найдите все возможные варианты. Обозначим выключенное состояние числом 0, а включённое — числом 1.

In [ ]:
x, y, z = Ints("switch_x switch_y switch_z")

goal = Goal()
goal.add(
    Or(x == 0, x == 1),
    Or(y == 0, y == 1),
    Or(z == 0, z == 1),
    x + y + z == 2
)

split_all = Repeat(
    OrElse(Tactic("split-clause"), Tactic("skip"))
)

solutions = []
for subgoal in split_all(goal):
    solver = Solver()
    solver.add(subgoal.as_expr())
    if solver.check() == sat:
        model = solver.model()
        solutions.append((model.eval(x), model.eval(y), model.eval(z)))

print("Возможные варианты:", solutions)


## Задача 3. Одна стратегия для двух типов задач

Создайте стратегию, которая сначала пытается разобрать логическую альтернативу, а если её нет — упрощает арифметическое выражение. Затем примените её к двум задачам:

1. `x` равен 2 или 5, но `x < 4`.
2. К неизвестному числу прибавили 2, затем ещё 3 и получили 10.

In [ ]:
def solve_with_strategy(*constraints):
    strategy = Then(
        OrElse(Tactic("split-clause"), Tactic("simplify")),
        Tactic("solve-eqs")
    ).solver()
    strategy.add(*constraints)
    print("Статус:", strategy.check())
    print("Модель:", strategy.model())

x = Int("strategy_x")
print("Задача 1:")
solve_with_strategy(Or(x == 2, x == 5), x < 4)

y = Int("strategy_y")
print("Задача 2:")
solve_with_strategy(y + 2 + 3 == 10)


## Задача 4. Порядок на пьедестале

На пьедестале стоят Маша, Оля и Катя. Маша не первая. Оля стоит левее Кати, а Маша стоит левее Кати. Определите порядок детей слева направо.

In [ ]:
masha, olya, katya = Ints("masha olya katya")

solver = Solver()
solver.add(
    And(
        1 <= masha, masha <= 3,
        1 <= olya, olya <= 3,
        1 <= katya, katya <= 3
    ),
    Distinct(masha, olya, katya),
    masha != 1,
    olya < katya,
    masha < katya
)

print("Статус:", solver.check())
model = solver.model()
print("Маша: место", model[masha])
print("Оля: место", model[olya])
print("Катя: место", model[katya])


## Задача 5. Три однозначных числа

На доске записаны три положительных однозначных числа `x`, `y` и `z`. Известно, что `x + y + z = 12`, а `z = 2x + y`. Найдите все три числа.

In [ ]:
bounded_solver = Then(
    With("simplify", arith_lhs=True, som=True),
    "normalize-bounds",
    "lia2pb",
    "pb2bv",
    "bit-blast",
    "sat"
).solver()

x, y, z = Ints("number_x number_y number_z")
bounded_solver.add(
    1 <= x, x <= 9,
    1 <= y, y <= 9,
    1 <= z, z <= 9,
    x + y + z == 12,
    z == 2 * x + y
)

print("Статус:", bounded_solver.check())
print("Модель:", bounded_solver.model())
